# Iniício da ingestão dos dados:

### Extração dos dados brutos do pdf

In [4]:
import pdfplumber
import pandas as pd
import re
import os

# Ingestão dos dados:

## Etapa 1: Preparação do Extrator (Regex e Máquina de Estado)
O primeiro passo do nosso pipeline de ETL é construir a lógica de extração. Como os PDFs da UERJ não possuem tabelas estruturadas padrão, utilizaremos Expressões Regulares (Regex) para identificar cirurgicamente as linhas de candidatos.

Para lidar com a quebra de páginas no meio da listagem, implementamos uma lógica de **Máquina de Estado**: o script rastreia o cabeçalho `CURSO:` e guarda essa informação na memória, associando cada candidato extraído ao curso correto, independentemente da página.

In [5]:
def extrair_dados_completos(caminho_pdf):
    # Regex 1: Identifica quando o curso muda
    padrao_curso = re.compile(r"^CURSO:\s+(?P<curso>.*)$")
    
    # Regex 2: Captura os dados estruturados do candidato
    padrao_candidato = re.compile(
        r"^(?P<nome>.*?)\s+"                   
        r"(?P<inscricao>\d{9}-\d)\s+"          
        r"(?P<subopcao>.*?)\s+"                
        r"(?P<classificacao_subopcao>\d+)\s+"  
        r"(?P<codigo>[A-Z/]+)\s+"              
        r"(?P<classificacao_final>\d+)$"       
    )

    dados_brutos = []
    curso_atual = "Desconhecido" # Variável de "memória" de estado

    with pdfplumber.open(caminho_pdf) as pdf:
        for pagina in pdf.pages:
            texto = pagina.extract_text()
            if texto:
                for linha in texto.split('\n'):
                    linha = linha.strip()
                    
                    # 1. Verifica se a linha dita um novo curso
                    match_curso = padrao_curso.match(linha)
                    if match_curso:
                        curso_atual = match_curso.group("curso").strip()
                        continue 
                        
                    # 2. Verifica se a linha é de um candidato
                    match_candidato = padrao_candidato.match(linha)
                    if match_candidato:
                        dados_brutos.append({
                            'curso': curso_atual,
                            'nome': match_candidato.group('nome').strip(),
                            'inscricao': match_candidato.group('inscricao').strip(),
                            'subopcao': match_candidato.group('subopcao').strip(),
                            'classificacao_na_subopcao': int(match_candidato.group('classificacao_subopcao')),
                            'codigo': match_candidato.group('codigo').strip(),
                            'classificacao_final': int(match_candidato.group('classificacao_final'))
                        })
                        
    return pd.DataFrame(dados_brutos)

## Etapa 2: Teste Unitário de Extração
Antes de processarmos toda a base de dados histórica, precisamos validar se o nosso extrator está funcionando corretamente em um único arquivo de amostra. Isso garante que não propagaremos erros ao longo do pipeline.

In [6]:
# Testando a extração com o arquivo do primeiro ano disponível
arquivo_teste = "Classificados/2018_classificados_UERJ.pdf"
print(f"Testando extração no arquivo: {arquivo_teste}")

df_teste = extrair_dados_completos(arquivo_teste)
display(df_teste.head())

Testando extração no arquivo: Classificados/2018_classificados_UERJ.pdf


,curso,nome,inscricao,subopcao,classificacao_na_subopcao,codigo,classificacao_final
0,Administração (RIO),ALESSANDRA TEIXEIRA DA COSTA,183005897-8,A2 - 2º semestre - Bacharelado - noite,136,N/I,2
1,Administração (RIO),ALESSANDRO MONTEIRO FERNANDES BRITO,183010412-0,A1 - 1º semestre - Bacharelado - manhã,37,NR,37
2,Administração (RIO),ANA CAROLINA FERNANDES DE LIMA,183007895-2,A2 - 2º semestre - Bacharelado - noite,45,NR,45
3,Administração (RIO),ANA CAROLINA PEREIRA DA SILVA,183009695-0,A1 - 1º semestre - Bacharelado - manhã,192,RP,8
4,Administração (RIO),ANA CLARA TAUIL BARENCO RIBEIRO,183006440-8,A2 - 2º semestre - Bacharelado - noite,21,NR,21


## Etapa 3: Ingestão em Lote (Batch Processing)
Com a função de extração validada, agora escalamos o processo. Este script itera automaticamente sobre todos os PDFs disponíveis na pasta `Classificados`. 

Durante a ingestão, já aplicamos uma transformação leve (Casting): extraímos o ano do nome do arquivo e forçamos a tipagem para inteiro, garantindo consistência para a futura modelagem no banco de dados.

In [7]:
pasta_pdfs = "Classificados"
lista_dfs = []
arquivos_pdf = [f for f in os.listdir(pasta_pdfs) if f.endswith('.pdf')]

print(f"Iniciando extração em lote de {len(arquivos_pdf)} arquivos...\n")

for arquivo in arquivos_pdf:
    caminho_completo = os.path.join(pasta_pdfs, arquivo)
    print(f"Processando: {arquivo}...")
    
    df_ano = extrair_dados_completos(caminho_completo)
    
    # Extrai o ano do arquivo e força a tipagem para inteiro
    df_ano['ano_vestibular'] = int(arquivo[:4])
    
    lista_dfs.append(df_ano)

# Consolida todos os anos em um único DataFrame
df_final = pd.concat(lista_dfs, ignore_index=True)
print("\nExtração finalizada com sucesso!")

Iniciando extração em lote de 9 arquivos...

Processando: 2018_classificados_UERJ.pdf...
Processando: 2019_classificados_UERJ.pdf...
Processando: 2020_classificados_UERJ.pdf...
Processando: 2021_classificados_UERJ.pdf...
Processando: 2022_classificados_UERJ.pdf...
Processando: 2023_classificados_UERJ.pdf...
Processando: 2024_classificados-UERJ.pdf...
Processando: 2025_classificados_UERJ.pdf...
Processando: 2026_classificados_UERJ.pdf...

Extração finalizada com sucesso!


## Etapa 4: Profiling Inicial dos Dados
Verificamos a saúde do DataFrame consolidado para garantir que não há valores nulos inesperados, que os tipos de dados estão corretos e que o volume de registros por ano faz sentido.

In [8]:
# Inspeção estrutural das colunas e tipagens
df_final.info()

<class 'pandas.DataFrame'>
RangeIndex: 45325 entries, 0 to 45324
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   curso                      45325 non-null  str  
 1   nome                       45325 non-null  str  
 2   inscricao                  45325 non-null  str  
 3   subopcao                   45325 non-null  str  
 4   classificacao_na_subopcao  45325 non-null  int64
 5   codigo                     45325 non-null  str  
 6   classificacao_final        45325 non-null  int64
 7   ano_vestibular             45325 non-null  int64
dtypes: int64(3), str(5)
memory usage: 2.8 MB


In [9]:
print("\n--- Distribuição de Registros por Ano ---")
display(df_final['ano_vestibular'].value_counts().sort_index())


--- Distribuição de Registros por Ano ---


ano_vestibular
2018    4746
2019    5238
2020    5441
2021    4645
2022    4705
2023    5292
2024    5000
2025    5119
2026    5139
Name: count, dtype: int64

## Etapa 5: Limpeza e Anonimização (Adequação LGPD)
Nesta etapa de transformação (Silver layer), aplicamos regras de negócio e de segurança:
1. **Anonimização (LGPD):** Retemos apenas o primeiro nome dos candidatos, removendo sobrenomes para evitar a identificação direta, mas mantendo o dado útil para contagens.
2. **Descarte:** Removemos as colunas de `inscricao` e `classificacao_final`, que não agregam valor analítico ao modelo relacional.
3. **Normalização:** Limpamos a coluna `subopcao` para remover os códigos internos da universidade e semestres, mantendo apenas a formação e o turno (ex: "Bacharelado - manhã").

In [10]:
# Fazemos uma cópia para não alterar o df_final original e manter o histórico do pipeline
df_limpo = df_final.copy()

# 1. Manter apenas o primeiro nome (quebra a string pelos espaços e pega a posição 0)
df_limpo['nome'] = df_limpo['nome'].str.split().str[0]

# 2. Excluir as colunas indesejadas
df_limpo = df_limpo.drop(columns=['inscricao'])

# 3. Limpar a Subopção
# Como a string original é "A2 - 2º semestre - Bacharelado - tarde/noite", 
# nós quebramos ela usando o separador " - ", pegamos os dois últimos blocos ([-2:]) 
# e juntamos de novo com o mesmo separador.
df_limpo['subopcao'] = df_limpo['subopcao'].str.split(' - ').str[-2:].str.join(' - ')

print("Transformação concluída. Amostra dos dados limpos:")
display(df_limpo.head(10))

Transformação concluída. Amostra dos dados limpos:


,curso,nome,subopcao,classificacao_na_subopcao,codigo,classificacao_final,ano_vestibular
0,Administração (RIO),ALESSANDRA,Bacharelado - noite,136,N/I,2,2018
1,Administração (RIO),ALESSANDRO,Bacharelado - manhã,37,NR,37,2018
2,Administração (RIO),ANA,Bacharelado - noite,45,NR,45,2018
3,Administração (RIO),ANA,Bacharelado - manhã,192,RP,8,2018
4,Administração (RIO),ANA,Bacharelado - noite,21,NR,21,2018
5,Administração (RIO),ANA,Bacharelado - noite,229,N/I,4,2018
6,Administração (RIO),ANA,Bacharelado - noite,10,NR,10,2018
7,Administração (RIO),ANA,Bacharelado - manhã,284,N/I,8,2018
8,Administração (RIO),ANA,Bacharelado - noite,147,RP,1,2018
9,Administração (RIO),ANDREA,Bacharelado - noite,219,RP,5,2018
